# Forecasting com XGBoost

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | XGBoost — Extreme Gradient Boosting (regressão de séries temporais via lags) |
| **Biblioteca** | `xgboost` 3.2.0 — `XGBRegressor`; `optuna` 4.8.0 (otimização bayesiana) |
| **Hiperparâmetros (espaço de busca)** | `max_depth` ∈ [2,7], `n_estimators` ∈ [50,400], `learning_rate` ∈ [0.01,0.3] (log), `subsample` ∈ [0.6,1.0], `colsample_bytree` ∈ [0.6,1.0], `min_child_weight` ∈ [1,10], `reg_alpha` ∈ [1e-3,1.0] (log), `reg_lambda` ∈ [0.5,5.0]; `N_LAGS=12` (fixo) |
| **Busca de hiperparâmetros** | Sim — Optuna TPE (Bayesiano), 50 trials por série (`N_TRIALS=50`), validação temporal nos últimos 20% do treino |
| **Critério de seleção** | MAPE (holdout temporal ~20% do treino) |
| **Séries utilizadas** | 29 séries com total ≥ 46 observações (`len(series) < N_LAGS + TEST_SIZE + 10`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses; 15 features: 12 lags + MA3 + MA6 + STD6 |
| **Reprodutibilidade** | `SEED = 42` (`random_state` no modelo + `TPESampler(seed=42)` + `np.random.seed(42)`) |
| **Referência** | Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *Proc. 22nd ACM SIGKDD*, 785–794. Akiba, T. et al. (2019). Optuna: A Next-generation Hyperparameter Optimization Framework. *KDD 2019*. |

---

## 1. Instalação de Dependências

In [1]:
# Instalação das dependências: XGBoost para o modelo e Optuna para otimização bayesiana
%pip install xgboost optuna arch -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importação de Bibliotecas

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import random

# ── Semente global para reprodutibilidade ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
import xgboost as xgb                    # Modelo de gradient boosting
import optuna                             # Otimização bayesiana de hiperparâmetros
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)  # Suprime logs do Optuna

# Carregar a base de dados com 29 séries econômicas brasileiras (108 meses cada)
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)
print(f"Shape: {df.shape} | Período: {df.index.min():%Y-%m} a {df.index.max():%Y-%m}")
print(f"Séries: {', '.join(df.columns)}")

C:\Users\phill\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (108, 35) | Período: 2017-01 a 2025-12
Séries: IBC_Br, Selic, Cambio_USDBRL, Desemprego, Brent_USD, Soja_USD, Minerio_USD, Ibovespa, ICC_FGV, Credito_Total, Inadimplencia, Massa_Salarial, CPI_USA, Prod_Ind_USA, Cafe_USD, Ouro_USD, GasNatural_USD, Cobre_USD, ETF_Emergentes, IGP_DI, INCC, ICE_Empresarial, Housing_Starts_EUA, Dollar_Index_Fed, PMS_Volume, PMC_Ampliado, IGPM, INPC, M2, Divida_PIB, Vendas_Varejo, Balanca_Comercial, NUCI_FGV, EAI_Emprego_Ind, SP500


## 3. Configuração do Experimento

In [3]:
# ============================================================
# Constantes do experimento
# ============================================================
N_LAGS = 12       # Janela de lags: últimos 12 meses como features de entrada
HORIZON = 3       # Horizonte de previsão: 3 meses à frente
TEST_SIZE = 24    # Período de teste: últimos 24 meses da base
SEED = 42         # Semente para reprodutibilidade
N_TRIALS = 50     # Número de tentativas do Optuna por série (busca bayesiana)


def create_features(series, n_lags=12):
    """
    Cria features univariadas para o XGBoost a partir de uma série temporal.
    
    Para cada ponto t, gera um vetor de 15 features:
      - 12 lags: [y(t-12), y(t-11), ..., y(t-1)]
      - MA3: média móvel dos últimos 3 meses (tendência de curto prazo)
      - MA6: média móvel dos últimos 6 meses (tendência de médio prazo)
      - STD6: desvio padrão dos últimos 6 meses (proxy de volatilidade)
    """
    X, y = [], []
    for i in range(n_lags, len(series)):
        lags = list(series[i-n_lags:i])
        ma3 = np.mean(series[max(0, i-3):i])
        ma6 = np.mean(series[max(0, i-6):i])
        std6 = np.std(series[max(0, i-6):i], ddof=1) if i >= 6 else 0.0
        X.append(lags + [ma3, ma6, std6])
        y.append(series[i])
    return np.array(X), np.array(y)


def pred_features(buffer, n_lags=12):
    """
    Constrói o vetor de features para previsão recursiva.
    
    Usa o buffer atualizado (que inclui previsões anteriores) para manter
    a mesma estrutura de features do treino. A cada passo, o buffer é
    estendido com a previsão anterior, permitindo prever h passos à frente.
    """
    s = np.array(buffer)
    lags = list(s[-n_lags:])
    ma3 = float(np.mean(s[-3:]))
    ma6 = float(np.mean(s[-6:]))
    std6 = float(np.std(s[-6:], ddof=1)) if len(s) >= 6 else 0.0
    return np.array(lags + [ma3, ma6, std6])


def calc_metrics(y_true, y_pred):
    """Calcula MAE, RMSE e MAPE. Usa epsilon 1e-8 para evitar divisão por zero no MAPE."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}


def tune_optuna(X_tr, y_tr, X_val, y_val, n_trials=50, seed=42):
    """
    Tuning bayesiano de hiperparâmetros com Optuna (TPE sampler).
    
    Busca no espaço de hiperparâmetros do XGBoost minimizando MAPE na
    validação temporal. O MAPE é usado como objetivo porque é a métrica
    principal de avaliação da dissertação.
    
    Espaço de busca:
      - max_depth: 2-7 (controla complexidade da árvore)
      - n_estimators: 50-400 (número de árvores no ensemble)
      - learning_rate: 0.01-0.3 (taxa de aprendizado, escala log)
      - subsample: 0.6-1.0 (fração de amostras por árvore)
      - colsample_bytree: 0.6-1.0 (fração de features por árvore)
      - min_child_weight: 1-10 (regularização por tamanho mínimo do nó)
      - reg_alpha: 1e-3 a 1.0 (regularização L1, escala log)
      - reg_lambda: 0.5-5.0 (regularização L2)
    """
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 2, 7),
            'n_estimators': trial.suggest_int('n_estimators', 50, 400),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 5.0),
            'objective': 'reg:squarederror',
            'verbosity': 0,
            'random_state': seed,
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        return np.mean(np.abs((y_val - preds) / (y_val + 1e-8))) * 100

    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    best.update({'objective': 'reg:squarederror', 'verbosity': 0, 'random_state': seed})
    return best


print(f"Configuração: N_LAGS={N_LAGS}, HORIZON={HORIZON}, TEST_SIZE={TEST_SIZE}")
print(f"Features: {N_LAGS} lags + MA3 + MA6 + STD6 = {N_LAGS + 3} features")
print(f"Optuna: {N_TRIALS} trials por série (TPE Bayesiano)")

Configuração: N_LAGS=12, HORIZON=3, TEST_SIZE=24
Features: 12 lags + MA3 + MA6 + STD6 = 15 features
Optuna: 50 trials por série (TPE Bayesiano)


## 4. Treinamento e Previsão (Walk-Forward)

In [ ]:
print("=" * 70)
print("XGBOOST OTIMIZADO — Tuning Bayesiano por Série + Walk-Forward")
print("=" * 70)

ALL_SERIES = list(df.columns)  # Lista com as 29 séries econômicas
# Excluir PIM e IPCA_mensal da análise
ALL_SERIES = [s for s in ALL_SERIES if s not in ['PIM', 'IPCA_mensal']]
all_results = []                # Armazena métricas e previsões de cada série
tuned_params = {}               # Guarda os hiperparâmetros otimizados por série

for idx, col in enumerate(ALL_SERIES, 1):
    series = df[col].dropna().values

    # Verificação de segurança: série precisa ter dados suficientes
    if len(series) < N_LAGS + TEST_SIZE + 10:
        print(f"[{idx}/{len(ALL_SERIES)}] {col} — série muito curta, pulando")
        continue

    # Separação temporal: treino (84 meses) e teste (24 meses finais)
    train_s = series[:-TEST_SIZE]
    test_s = series[-TEST_SIZE:]

    # ============================================================
    # FASE 1: Tuning Bayesiano (Optuna) — busca os melhores hiperparâmetros
    # Usa apenas dados de treino com validação temporal (80% treino / 20% validação)
    # ============================================================
    X_all, y_all = create_features(train_s, N_LAGS)

    # Normalização: StandardScaler em X e y (melhora convergência do XGBoost)
    sc_X_tune = StandardScaler()
    sc_y_tune = StandardScaler()
    X_sc = sc_X_tune.fit_transform(X_all)
    y_sc = sc_y_tune.fit_transform(y_all.reshape(-1, 1)).flatten()

    # Validação temporal: últimos 20% dos dados de treino (mínimo 5 pontos)
    val_n = max(5, int(len(X_sc) * 0.2))
    best_p = tune_optuna(X_sc[:-val_n], y_sc[:-val_n],      # sub-treino
                         X_sc[-val_n:], y_sc[-val_n:],        # validação
                         n_trials=N_TRIALS, seed=SEED)
    tuned_params[col] = best_p  # Salva parâmetros otimizados para esta série

    # ============================================================
    # FASE 2: Walk-Forward Validation — avaliação nos 24 meses de teste
    # A cada janela de 3 meses, retreina o modelo com todos os dados até ali
    # ============================================================
    preds_all, actuals_all = [], []

    for i in range(0, len(test_s), HORIZON):
        # Expansão progressiva: dados de treino + dados de teste já "revelados"
        cur_train = np.concatenate([train_s, test_s[:i]]) if i > 0 else train_s
        n_steps = min(HORIZON, len(test_s) - i)  # Última janela pode ser menor que 3

        # Recria features e escala com o treino expandido
        X_tr, y_tr = create_features(cur_train, N_LAGS)
        sx, sy = StandardScaler(), StandardScaler()
        X_tr_s = sx.fit_transform(X_tr)
        y_tr_s = sy.fit_transform(y_tr.reshape(-1, 1)).flatten()

        # Treina novo modelo com os hiperparâmetros otimizados da Fase 1
        model = xgb.XGBRegressor(**best_p)
        model.fit(X_tr_s, y_tr_s)

        # Previsão recursiva: cada previsão alimenta a próxima
        buf = list(cur_train)  # Buffer com histórico completo para gerar features
        for h in range(n_steps):
            feat = pred_features(buf, N_LAGS)       # Gera features a partir do buffer
            yp_s = model.predict(sx.transform(feat.reshape(1, -1)))  # Predição normalizada
            yp = sy.inverse_transform(yp_s.reshape(-1, 1)).flatten()[0]  # Desnormaliza
            preds_all.append(yp)
            buf.append(yp)  # Adiciona previsão ao buffer para o próximo passo

        actuals_all.extend(test_s[i:i+n_steps])

    # Calcula métricas agregadas para toda a janela de teste (24 meses)
    preds_arr = np.array(preds_all)
    actuals_arr = np.array(actuals_all)
    m = calc_metrics(actuals_arr, preds_arr)

    # Imprime resultado com os 3 hiperparâmetros mais informativos
    print(f"[{idx:2}/{len(ALL_SERIES)}] {col:20} | MAE={m['MAE']:.4f} | MAPE={m['MAPE']:.2f}%"
          f"  (depth={best_p['max_depth']}, n_est={best_p['n_estimators']}, lr={best_p['learning_rate']:.3f})")

    all_results.append({
        'Serie': col, 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'MAPE': m['MAPE'],
        'N_Pontos': len(preds_arr), 'preds': preds_arr, 'actuals': actuals_arr
    })

print(f"\n{'=' * 70}")
print(f"Concluído: {len(all_results)} séries processadas")
print("=" * 70)

XGBOOST OTIMIZADO — Tuning Bayesiano por Série + Walk-Forward
[ 1/35] IBC_Br               | MAE=2.3406 | MAPE=2.12%  (depth=7, n_est=198, lr=0.037)
[ 2/35] Selic                | MAE=3.7201 | MAPE=25.80%  (depth=6, n_est=50, lr=0.022)
[ 3/35] Cambio_USDBRL        | MAE=0.1402 | MAPE=2.54%  (depth=3, n_est=309, lr=0.031)
[ 4/35] Desemprego           | MAE=0.4368 | MAPE=7.63%  (depth=6, n_est=305, lr=0.054)
[ 5/35] Brent_USD            | MAE=5.7484 | MAPE=8.70%  (depth=4, n_est=73, lr=0.013)
[ 6/35] Soja_USD             | MAE=19.5185 | MAPE=5.18%  (depth=4, n_est=68, lr=0.013)
[ 7/35] Minerio_USD          | MAE=6.6389 | MAPE=6.52%  (depth=2, n_est=68, lr=0.010)
[ 8/35] Ibovespa             | MAE=11539.9329 | MAPE=7.93%  (depth=4, n_est=264, lr=0.299)
[ 9/35] ICC_FGV              | MAE=6.0744 | MAPE=5.15%  (depth=7, n_est=54, lr=0.294)
[10/35] Credito_Total        | MAE=53591.7292 | MAPE=1.36%  (depth=7, n_est=352, lr=0.066)
[11/35] Inadimplencia        | MAE=0.4005 | MAPE=8.12%  (depth=

## 5. Resultados e Métricas

In [5]:
# Monta DataFrame de resultados (sem arrays de previsões) ordenado por MAPE
df_res = pd.DataFrame([{k: v for k, v in r.items() if k not in ('preds', 'actuals')}
                       for r in all_results]).sort_values('MAPE')

print("=" * 60)
print("RESULTADOS — XGBoost Otimizado (Bayesiano)")
print("=" * 60)
print(f"\nMAPE médio: {df_res['MAPE'].mean():.2f}%  |  MAE médio: {df_res['MAE'].mean():.4f}  |  RMSE médio: {df_res['RMSE'].mean():.4f}\n")

# Ranking com medalhas para as 3 melhores séries
for i, (_, r) in enumerate(df_res.iterrows(), 1):
    tag = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{tag} {i:2}. {r['Serie']:20} MAPE={r['MAPE']:7.2f}%  MAE={r['MAE']:.4f}")

RESULTADOS — XGBoost Otimizado (Bayesiano)

MAPE médio: 11434519.38%  |  MAE médio: 5759757.7058  |  RMSE médio: 6872557.1463

🥇  1. CPI_USA              MAPE=   0.48%  MAE=1.5392
🥈  2. Prod_Ind_USA         MAPE=   0.72%  MAE=0.7325
🥉  3. Massa_Salarial       MAPE=   1.28%  MAE=4604.8464
    4. Credito_Total        MAPE=   1.36%  MAE=53591.7292
    5. M2                   MAPE=   1.41%  MAE=201127292.3333
    6. PMS_Volume           MAPE=   1.46%  MAE=1.6181
    7. Divida_PIB           MAPE=   1.49%  MAE=0.9550
    8. NUCI_FGV             MAPE=   1.50%  MAE=1.2198
    9. IBC_Br               MAPE=   2.12%  MAE=2.3406
   10. Cambio_USDBRL        MAPE=   2.54%  MAE=0.1402
   11. Dollar_Index_Fed     MAPE=   2.70%  MAE=3.3323
   12. PMC_Ampliado         MAPE=   3.14%  MAE=0.0189
   13. Vendas_Varejo        MAPE=   3.19%  MAE=3.6146
   14. ICE_Empresarial      MAPE=   3.52%  MAE=3.6327
   15. Housing_Starts_EUA   MAPE=   5.02%  MAE=66.8018
   16. ICC_FGV              MAPE=   5.15%  MAE=6.0

## 6. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = df_res.sort_values('MAPE')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df['Serie'])
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'XGBoost — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('xgboost_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: xgboost_mape_por_serie.png")

## 7. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df['Serie'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    r = next(x for x in all_results if x['Serie'] == sn)

    # Valores reais (teste)
    ax.plot(r['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(r['preds'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {r['MAPE']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('XGBoost — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('xgboost_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: xgboost_previsoes.png")

## 8. Exportação de Resultados

In [ ]:
# ── Exportação dos Resultados ──# DataFrame de métricas (sem arrays)results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('preds', 'actuals')}                           for r in all_results]).sort_values('MAPE')results_df.to_csv('resultados_xgboost.csv', index=False)print(f"✅ Salvo: resultados_xgboost.csv ({len(results_df)} séries)")# DataFrame de previsões detalhadasrows = []for r in all_results:    for j, (real, pred) in enumerate(zip(r['actuals'], r['preds'])):        rows.append({'Serie': r['Serie'], 'Step': j+1, 'Real': real, 'Previsto': pred})df_prev = pd.DataFrame(rows)df_prev.to_csv('previsoes_xgboost.csv', index=False)print(f"✅ Salvo: previsoes_xgboost.csv ({len(df_prev)} linhas)")